# Análisis Exploratorio - Vitrina Escolar Chile

**Fecha de datos**: 2026-08-02  
**Dataset**: 7,673 establecimientos | 11 comunas procesadas | 146 RBDs únicos

## Ejes de Análisis
1. Oferta Educativa y Cobertura Geográfica
2. Capacidad, Demanda y Vacantes
3. Calidad y Resultados (SIMCE + Desarrollo Personal)
4. Segregación y Equidad
5. Infraestructura y Actividades

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

DATA_PATH = Path.cwd() / 'data' / 'processed' / '2026-08-02'

if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / 'data' / 'processed' / '2026-08-02'

In [ ]:
# Carga de datos
establecimientos = pq.read_table(f'{DATA_PATH}/establecimientos.parquet').to_pandas()
sedes = pq.read_table(f'{DATA_PATH}/sedes.parquet').to_pandas()
cursos = pq.read_table(f'{DATA_PATH}/cursos.parquet').to_pandas()
actividades = pq.read_table(f'{DATA_PATH}/actividades.parquet').to_pandas()
indicadores = pq.read_table(f'{DATA_PATH}/indicadores.parquet').to_pandas()
imagenes = pq.read_table(f'{DATA_PATH}/imagenes.parquet').to_pandas()

print(f'Establecimientos: {len(establecimientos):,}')
print(f'Sedes: {len(sedes):,}')
print(f'Cursos: {len(cursos):,}')
print(f'Actividades: {len(actividades):,}')
print(f'Indicadores: {len(indicadores):,}')
print(f'Imágenes: {len(imagenes):,}')

---
## Eje 1: Oferta Educativa y Cobertura Geográfica

Analizamos la distribución de establecimientos por dependencia, región y comuna, identificando zonas con déficit de cobertura.

In [ ]:
# 1.1 Distribución por dependencia
fig = px.pie(
    establecimientos['dependencia'].value_counts().reset_index(),
    values='count',
    names='dependencia',
    title='Distribución de Establecimientos por Dependencia',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# 1.2 Establecimientos por región
sedes_por_region = sedes.groupby('region')['rbd'].nunique().reset_index()
sedes_por_region.columns = ['region', 'establecimientos']
sedes_por_region = sedes_por_region.sort_values('establecimientos', ascending=True)

fig = px.bar(
    sedes_por_region,
    x='establecimientos',
    y='region',
    orientation='h',
    title='Establecimientos Únicos por Región',
    labels={'establecimientos': 'N° Establecimientos', 'region': 'Región'}
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# 1.3 Top 20 comunas con más establecimientos
sedes_por_comuna = sedes.groupby(['region', 'comuna'])['rbd'].nunique().reset_index()
sedes_por_comuna.columns = ['region', 'comuna', 'establecimientos']
top_comunas = sedes_por_comuna.nlargest(20, 'establecimientos')

fig = px.bar(
    top_comunas,
    x='comuna',
    y='establecimientos',
    color='region',
    title='Top 20 Comunas por Cantidad de Establecimientos',
    labels={'establecimientos': 'N° Establecimientos', 'comuna': 'Comuna'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# 1.4 Cobertura por nivel educativo
niveles_por_dependencia = establecimientos.groupby(['dependencia', 'nivel_maximo']).size().reset_index(name='count')

fig = px.bar(
    niveles_por_dependencia,
    x='nivel_maximo',
    y='count',
    color='dependencia',
    barmode='group',
    title='Cobertura por Nivel Máximo y Dependencia',
    labels={'count': 'N° Establecimientos', 'nivel_maximo': 'Nivel Máximo'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# 1.5 Establecimientos únicos en su comuna (potencial monopolio)
cursos_unicos = cursos[cursos['unico_comuna'] == True]
unicos_por_comuna = cursos_unicos.merge(sedes[['rbd', 'codigo_sede', 'comuna']], on=['rbd', 'codigo_sede'])
monopolios = unicos_por_comuna.groupby('comuna')['codigo_curso'].nunique().reset_index()
monopolios.columns = ['comuna', 'cursos_unicos']
monopolios = monopolios.nlargest(15, 'cursos_unicos')

fig = px.bar(
    monopolios,
    x='comuna',
    y='cursos_unicos',
    title='Comunas con Más Cursos de Establecimientos Únicos (Sin Competencia)',
    labels={'cursos_unicos': 'N° Cursos Únicos', 'comuna': 'Comuna'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Eje 2: Capacidad, Demanda y Vacantes

Analizamos ratios alumno/docente, presión de vacantes y dinámicas de pre-inscripción.

In [ ]:
# 2.1 Ratio alumnos/docente por dependencia
establecimientos['ratio_alumno_docente'] = establecimientos['alumnos_matriculados'] / establecimientos['cantidad_docentes'].replace(0, 1)

fig = px.box(
    establecimientos,
    x='dependencia',
    y='ratio_alumno_docente',
    title='Ratio Alumnos/Docente por Dependencia',
    labels={'ratio_alumno_docente': 'Ratio Alumnos/Docente', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 2.2 Distribución de matrícula
fig = px.histogram(
    establecimientos[establecimientos['alumnos_matriculados'] > 0],
    x='alumnos_matriculados',
    color='dependencia',
    marginal='box',
    nbins=50,
    title='Distribución de Matrícula por Establecimiento',
    labels={'alumnos_matriculados': 'Alumnos Matriculados', 'count': 'Frecuencia'}
)
fig.show()

In [ ]:
# 2.3 Vacantes por nivel educativo
vacantes_por_nivel = cursos.groupby('glosa_grupo_ensenanza').agg({
    'cupos_totales': 'sum',
    'vacantes_rango_inferior': 'sum',
    'vacantes_rango_superior': 'sum'
}).reset_index()

vacantes_por_nivel['tasa_vacantes_inf'] = vacantes_por_nivel['vacantes_rango_inferior'] / vacantes_por_nivel['cupos_totales'] * 100
vacantes_por_nivel['tasa_vacantes_sup'] = vacantes_por_nivel['vacantes_rango_superior'] / vacantes_por_nivel['cupos_totales'] * 100

fig = go.Figure()
fig.add_trace(go.Bar(name='Vacantes Rango Inferior (%)', x=vacantes_por_nivel['glosa_grupo_ensenanza'], y=vacantes_por_nivel['tasa_vacantes_inf']))
fig.add_trace(go.Bar(name='Vacantes Rango Superior (%)', x=vacantes_por_nivel['glosa_grupo_ensenanza'], y=vacantes_por_nivel['tasa_vacantes_sup']))
fig.update_layout(barmode='group', title='Tasa de Vacantes por Nivel Educativo (%)', yaxis_title='% Vacantes sobre Cupos')
fig.show()

In [ ]:
# 2.4 Tasa de repitencia por nivel
cursos['tasa_repitencia'] = cursos['repitentes_anio_actual'] / cursos['cupos_totales'].replace(0, 1) * 100

repitencia_por_nivel = cursos.groupby('glosa_nivel')['tasa_repitencia'].mean().reset_index()
repitencia_por_nivel = repitencia_por_nivel.sort_values('tasa_repitencia', ascending=False).head(15)

fig = px.bar(
    repitencia_por_nivel,
    x='glosa_nivel',
    y='tasa_repitencia',
    title='Top 15 Niveles con Mayor Tasa de Repitencia Promedio',
    labels={'tasa_repitencia': 'Tasa Repitencia (%)', 'glosa_nivel': 'Nivel'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# 2.5 Pre-inscritos vs vacantes (presión de demanda)
cursos['presion_demanda'] = cursos['pre_inscritos_anio_siguiente'] / cursos['pre_vacantes_inferior'].replace(0, 1)

presion_por_nivel = cursos.groupby('glosa_grupo_ensenanza')['presion_demanda'].mean().reset_index()

fig = px.bar(
    presion_por_nivel,
    x='glosa_grupo_ensenanza',
    y='presion_demanda',
    title='Presión de Demanda por Nivel Educativo (Pre-inscritos / Vacantes)',
    labels={'presion_demanda': 'Ratio Pre-inscritos/Vacantes', 'glosa_grupo_ensenanza': 'Nivel'}
)
fig.add_hline(y=1, line_dash='dash', line_color='red', annotation_text='Equilibrio')
fig.show()

---
## Eje 3: Calidad y Resultados (SIMCE + Desarrollo Personal)

Analizamos indicadores de calidad, benchmarking por dependencia y comparación con grupo socioeconómico (GSE).

In [ ]:
# 3.1 Distribución de puntajes por tipo de indicador
fig = px.box(
    indicadores,
    x='tipo_indicador',
    y='puntaje',
    title='Distribución de Puntajes por Tipo de Indicador',
    labels={'puntaje': 'Puntaje', 'tipo_indicador': 'Tipo'}
)
fig.show()

In [ ]:
# 3.2 Puntaje SIMCE por nivel
simce = indicadores[indicadores['tipo_indicador'] == 'SIMCE']

fig = px.box(
    simce,
    x='nivel_indicador',
    y='puntaje',
    title='Distribución Puntaje SIMCE por Nivel',
    labels={'puntaje': 'Puntaje SIMCE', 'nivel_indicador': 'Nivel'}
)
fig.show()

In [ ]:
# 3.3 Comparación GSE - ¿Rinden acorde a su grupo socioeconómico?
comparacion_gse = indicadores['comparacion_gse_glosa'].value_counts().reset_index()
comparacion_gse.columns = ['comparacion', 'count']
comparacion_gse = comparacion_gse[comparacion_gse['comparacion'].isin(['Similar', 'Más alto', 'Más bajo'])]

fig = px.pie(
    comparacion_gse,
    values='count',
    names='comparacion',
    title='Resultados vs Grupo Socioeconómico (GSE)',
    hole=0.4,
    color='comparacion',
    color_discrete_map={'Más alto': 'green', 'Similar': 'blue', 'Más bajo': 'red'}
)
fig.show()

In [ ]:
# 3.4 Puntaje por dependencia (join con establecimientos)
indicadores_est = indicadores.merge(establecimientos[['rbd', 'dependencia']], on='rbd')

fig = px.box(
    indicadores_est[indicadores_est['tipo_indicador'] == 'SIMCE'],
    x='dependencia',
    y='puntaje',
    title='Puntaje SIMCE por Dependencia',
    labels={'puntaje': 'Puntaje SIMCE', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 3.5 Desarrollo Personal - dimensiones
desarrollo = indicadores[indicadores['tipo_indicador'] == 'DESARROLLO_PERSONAL']

fig = px.box(
    desarrollo,
    x='nombre_indicador',
    y='puntaje',
    title='Indicadores de Desarrollo Personal - Puntajes por Dimensión',
    labels={'puntaje': 'Puntaje', 'nombre_indicador': 'Dimensión'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Eje 4: Segregación y Equidad

Analizamos copago, segregación por género, inclusión (PIE) y el rol de colegios religiosos.

In [ ]:
# 4.1 Distribución de copago por dependencia
cursos_est = cursos.merge(establecimientos[['rbd', 'dependencia']], on='rbd')
cursos_con_copago = cursos_est[cursos_est['copago_valor'] > 0]

fig = px.box(
    cursos_con_copago,
    x='dependencia',
    y='copago_valor',
    title='Distribución de Copago por Dependencia (solo con copago > 0)',
    labels={'copago_valor': 'Copago ($)', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 4.2 Proporción de cursos con copago por dependencia
copago_por_dep = cursos_est.groupby('dependencia').agg(
    total_cursos=('codigo_curso', 'count'),
    con_copago=('copago_valor', lambda x: (x > 0).sum())
).reset_index()
copago_por_dep['pct_con_copago'] = copago_por_dep['con_copago'] / copago_por_dep['total_cursos'] * 100

fig = px.bar(
    copago_por_dep,
    x='dependencia',
    y='pct_con_copago',
    title='% de Cursos con Copago por Dependencia',
    labels={'pct_con_copago': '% Cursos con Copago', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 4.3 Segregación por género
genero = establecimientos['regimen'].value_counts().reset_index()
genero.columns = ['regimen', 'count']

fig = px.pie(
    genero,
    values='count',
    names='regimen',
    title='Distribución por Régimen de Género',
    hole=0.4
)
fig.show()

In [ ]:
# 4.4 Inclusión (PIE) y Subvención Preferencial
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], subplot_titles=('Programa Integración Escolar (PIE)', 'Subvención Preferencial'))

pie_counts = establecimientos['integracion'].value_counts()
fig.add_trace(go.Pie(labels=['Con PIE', 'Sin PIE'], values=pie_counts.values, hole=0.4), row=1, col=1)

sep_counts = establecimientos['subvencion_preferencial'].value_counts()
fig.add_trace(go.Pie(labels=['Con SEP', 'Sin SEP'], values=sep_counts.values, hole=0.4), row=1, col=2)

fig.update_layout(height=400)
fig.show()

In [ ]:
# 4.5 Orientación religiosa y dependencia
religion_dep = establecimientos.groupby(['orientacion_religiosa', 'dependencia']).size().reset_index(name='count')
top_religiones = establecimientos['orientacion_religiosa'].value_counts().head(5).index
religion_dep_top = religion_dep[religion_dep['orientacion_religiosa'].isin(top_religiones)]

fig = px.bar(
    religion_dep_top,
    x='orientacion_religiosa',
    y='count',
    color='dependencia',
    barmode='group',
    title='Orientación Religiosa por Dependencia (Top 5)',
    labels={'count': 'N° Establecimientos', 'orientacion_religiosa': 'Orientación'}
)
fig.show()

---
## Eje 5: Infraestructura y Actividades

Analizamos la oferta de actividades extraprogramáticas, infraestructura, idiomas y cobertura visual.

In [ ]:
# 5.1 Actividades por tipo
act_por_tipo = actividades['tipo'].value_counts().reset_index()
act_por_tipo.columns = ['tipo', 'count']

fig = px.bar(
    act_por_tipo,
    x='tipo',
    y='count',
    title='Actividades por Tipo',
    labels={'count': 'N° Actividades', 'tipo': 'Tipo'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# 5.2 Actividades por dependencia
act_est = actividades.merge(establecimientos[['rbd', 'dependencia']], on='rbd')
act_por_dep = act_est.groupby(['dependencia', 'tipo']).size().reset_index(name='count')

fig = px.bar(
    act_por_dep,
    x='dependencia',
    y='count',
    color='tipo',
    barmode='group',
    title='Actividades por Tipo y Dependencia',
    labels={'count': 'N° Actividades', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 5.3 Actividades promedio por establecimiento
act_por_rbd = actividades.groupby('rbd').size().reset_index(name='total_actividades')
act_por_rbd = act_por_rbd.merge(establecimientos[['rbd', 'dependencia']], on='rbd')

fig = px.box(
    act_por_rbd,
    x='dependencia',
    y='total_actividades',
    title='Total de Actividades por Establecimiento y Dependencia',
    labels={'total_actividades': 'N° Actividades', 'dependencia': 'Dependencia'}
)
fig.show()

In [ ]:
# 5.4 Top actividades de infraestructura
infra = actividades[actividades['tipo'] == 'INFRAESTRUCTURA']
top_infra = infra['nombre'].value_counts().head(15).reset_index()
top_infra.columns = ['infraestructura', 'count']

fig = px.bar(
    top_infra,
    y='infraestructura',
    x='count',
    orientation='h',
    title='Top 15 Elementos de Infraestructura',
    labels={'count': 'N° Establecimientos', 'infraestructura': 'Infraestructura'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# 5.5 Idiomas ofrecidos
idiomas = actividades[actividades['tipo'] == 'IDIOMA']
idiomas_counts = idiomas['nombre'].value_counts().head(10).reset_index()
idiomas_counts.columns = ['idioma', 'count']

fig = px.bar(
    idiomas_counts,
    x='idioma',
    y='count',
    title='Idiomas Ofrecidos por Establecimientos',
    labels={'count': 'N° Establecimientos', 'idioma': 'Idioma'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# 5.6 Cobertura visual (imágenes por establecimiento)
img_por_rbd = imagenes.groupby('rbd').size().reset_index(name='total_imagenes')
img_por_rbd = img_por_rbd.merge(establecimientos[['rbd', 'dependencia']], on='rbd')

fig = px.box(
    img_por_rbd,
    x='dependencia',
    y='total_imagenes',
    title='Imágenes por Establecimiento y Dependencia',
    labels={'total_imagenes': 'N° Imágenes', 'dependencia': 'Dependencia'}
)
fig.show()

---
## Bonus: Mapa Geoespacial

Visualización de sedes en mapa con coordenadas.

In [ ]:
# Mapa de sedes (muestra de 1000 puntos para performance)
sedes_mapa = sedes.dropna(subset=['latitud', 'longitud']).sample(min(1000, len(sedes)), random_state=42)
sedes_mapa = sedes_mapa.merge(establecimientos[['rbd', 'dependencia']], on='rbd')

fig = px.scatter_mapbox(
    sedes_mapa,
    lat='latitud',
    lon='longitud',
    color='dependencia',
    hover_data=['comuna', 'rbd'],
    zoom=5,
    height=600,
    title='Distribución Geográfica de Sedes (Muestra)'
)
fig.update_layout(mapbox_style='open-street-map')
fig.show()

---
## Próximos Pasos

- [ ] Clustering no supervisado de establecimientos
- [ ] Score de accesibilidad por comuna
- [ ] Dashboard interactivo en Streamlit
- [ ] Análisis de correlación calidad ↔ demanda